# 05 — Validación de robustez y análisis de incertidumbre

## Objetivo

Evaluar la estabilidad del Precio Esperado Reciclado (PER) ante cambios en la selección de comparables y analizar las limitaciones asociadas a la clasificación del estado de las propiedades.

El análisis responde a las observaciones recibidas en la PreEntrega 2 y busca fortalecer la confiabilidad de las estimaciones utilizadas para identificar oportunidades de flipping inmobiliario en CABA.

### Preguntas de análisis

1. ¿Cuánto cambia el PER al excluir propiedades con señales contradictorias sobre su estado?
2. ¿Qué impacto tiene esa exclusión sobre las estimaciones y la identificación de oportunidades?
3. ¿Cómo afecta la inclusión de referencias con señales débiles de reciclado a la estabilidad del PER y a la selección de oportunidades?
4. ¿Qué diferencias presentan las propiedades sin clasificar respecto de las clasificadas y qué implicancias tienen para la representatividad de los comparables?

Se conserva el modelo original de la notebook 03 como referencia. Las modificaciones se evalúan como escenarios alternativos, sin sobrescribir los resultados originales.

In [5]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path.cwd().resolve()
while RAIZ != RAIZ.parent and not (RAIZ / "src").is_dir():
    RAIZ = RAIZ.parent
if not (RAIZ / "src").is_dir():
    raise FileNotFoundError("Abrí la notebook desde la carpeta del repositorio (que contiene src/).")
sys.path.insert(0, str(RAIZ))
from src.benchmark import benchmark_jerarquico, NIVELES_COMPARABLES
from src.config import MIN_COMPARABLES
ruta = RAIZ / "data/processed/remax_deptos_features.csv"
if not ruta.exists():
    raise FileNotFoundError(f"Falta {ruta}. Primero generá la base ejecutando notebook 03.")
df = pd.read_csv(ruta)
print(f"Base cargada: {df.shape[0]} filas, {df.shape[1]} columnas")

Base cargada: 9499 filas, 100 columnas


In [10]:

# 1. CONTROLES INICIALES

import pandas as pd
import numpy as np

print("=== BASE DE DATOS ===")
print("Filas:", len(df))
print("Columnas:", len(df.columns))
print("IDs duplicados:", df["ID"].duplicated().sum())

print("\n=== ESTADO DE LAS PROPIEDADES ===")
display(
    df["Estado_Propiedad"]
    .value_counts(dropna=False)
    .to_frame("Cantidad")
)

print("\n=== PER ORIGINAL ===")

mask_obra = df["Estado_Propiedad"].eq("Necesita_obra")
mask_per = mask_obra & df["PER_Base"].notna()

print("Propiedades que necesitan obra:", int(mask_obra.sum()))
print("Propiedades con PER:", int(mask_per.sum()))
print(
    "Cobertura PER:",
    round(mask_per.sum() / mask_obra.sum() * 100, 2),
    "%"
)

print("\n=== CONTRADICCIONES ===")
print(
    "Propiedades con señales contradictorias:",
    int(df["Flag_Senal_Contradictoria"].fillna(False).sum())
)

print("\n=== VALIDACIÓN CONTRA NOTEBOOK 03 ===")

controles = {
    "Base de 9499 propiedades": len(df) == 9499,
    "439 propiedades necesitan obra": mask_obra.sum() == 439,
    "428 propiedades tienen PER": mask_per.sum() == 428,
    "78 propiedades contradictorias":
        df["Flag_Senal_Contradictoria"].fillna(False).sum() == 78
}

for nombre, cumple in controles.items():
    print(("OK" if cumple else "REVISAR"), "-", nombre)


=== BASE DE DATOS ===
Filas: 9499
Columnas: 100
IDs duplicados: 0

=== ESTADO DE LAS PROPIEDADES ===


,Cantidad
Estado_Propiedad,
Sin_clasificar,6034
Buen_estado,2049
Reciclada_refaccionada,977
Necesita_obra,439



=== PER ORIGINAL ===
Propiedades que necesitan obra: 439
Propiedades con PER: 428
Cobertura PER: 97.49 %

=== CONTRADICCIONES ===
Propiedades con señales contradictorias: 78

=== VALIDACIÓN CONTRA NOTEBOOK 03 ===
OK - Base de 9499 propiedades
OK - 439 propiedades necesitan obra
OK - 428 propiedades tienen PER
OK - 78 propiedades contradictorias



## 2. Exclusión de referencias con señales contradictorias

Se evalúa el efecto de excluir del conjunto de referencias del PER aquellas propiedades que presentan señales contradictorias sobre su estado.

Se mantiene el mismo universo objetivo y la misma metodología del modelo original. La única modificación del escenario alternativo es la exclusión de estas referencias.

Esto permite aislar su impacto sobre la cobertura y las estimaciones del PER.


In [11]:

# 2.1 — Construcción de los conjuntos de referencia

flag_contradictoria = (
    df["Flag_Senal_Contradictoria"]
    .fillna(False)
    .astype(bool)
)

# Mismos filtros de selección de referencias que en la notebook 03
mask_ref_original = (
    df["Estado_Agrupado"].eq("Buen_o_reciclada")
    & ~df["Reciclado_Parcial"].fillna(False).astype(bool)
    & ~df["Flag_Extremo"].fillna(False).astype(bool)
    & ~df["Flag_Ambientes_Dudoso"].fillna(False).astype(bool)
)

# Escenario alternativo: excluir señales contradictorias
mask_ref_sin_contradicciones = (
    mask_ref_original & ~flag_contradictoria
)

# Universo objetivo: propiedades que necesitan obra
mask_objetivo_per = df["Estado_Propiedad"].eq("Necesita_obra")

resumen_referencias = pd.DataFrame({
    "Indicador": [
        "Referencias originales",
        "Referencias contradictorias excluidas",
        "Referencias sin contradicciones",
        "Porcentaje excluido"
    ],
    "Valor": [
        int(mask_ref_original.sum()),
        int((mask_ref_original & flag_contradictoria).sum()),
        int(mask_ref_sin_contradicciones.sum()),
        round(
            100 * (mask_ref_original & flag_contradictoria).sum()
            / mask_ref_original.sum(), 2
        )
    ]
})

display(resumen_referencias)

print("\nReferencias excluidas por barrio:")
display(
    df.loc[
        mask_ref_original & flag_contradictoria,
        "Barrio_Oficial"
    ]
    .value_counts()
    .to_frame("Cantidad")
)


,Indicador,Valor
0,Referencias originales,2844.00
1,Referencias contradictorias excluidas,10.00
2,Referencias sin contradicciones,2834.00
3,Porcentaje excluido,0.35



Referencias excluidas por barrio:


,Cantidad
Barrio_Oficial,
Caballito,2
Colegiales,2
San Cristobal,1
Flores,1
Villa Gral. Mitre,1
Recoleta,1
San Nicolas,1
Almagro,1


In [12]:

# 2.2 — Recalcular el PER excluyendo referencias contradictorias

from src.benchmark import benchmark_jerarquico, NIVELES_COMPARABLES
from src.config import MIN_COMPARABLES

# Reproducir el PER original para verificar consistencia
per_control = benchmark_jerarquico(
    df,
    valor="USD_m2_Homogeneizado",
    niveles=NIVELES_COMPARABLES,
    mask_referencia=mask_ref_original,
    mask_objetivo=mask_objetivo_per,
    min_n=MIN_COMPARABLES,
    leave_one_out=False
)

# Escenario alternativo: sin referencias contradictorias
per_sin_contradicciones = benchmark_jerarquico(
    df,
    valor="USD_m2_Homogeneizado",
    niveles=NIVELES_COMPARABLES,
    mask_referencia=mask_ref_sin_contradicciones,
    mask_objetivo=mask_objetivo_per,
    min_n=MIN_COMPARABLES,
    leave_one_out=False
)

# Construir tabla de comparación para propiedades que necesitan obra
comparacion_per = df.loc[mask_objetivo_per, [
    "ID",
    "Barrio_Oficial",
    "Ambientes",
    "Superficie_Homogeneizada_m2",
    "PER_Base"
]].copy()

comparacion_per["PER_Control"] = (
    per_control.loc[mask_objetivo_per, "Q50"]
    * df.loc[mask_objetivo_per, "Superficie_Homogeneizada_m2"]
)

comparacion_per["PER_Sin_Contradicciones"] = (
    per_sin_contradicciones.loc[mask_objetivo_per, "Q50"]
    * df.loc[mask_objetivo_per, "Superficie_Homogeneizada_m2"]
)

# Validar que reproducimos el PER original
diferencia_control = (
    comparacion_per["PER_Control"]
    - comparacion_per["PER_Base"]
).abs()

print("=== CONTROL DE REPRODUCIBILIDAD ===")
print("PER original disponible:",
      int(comparacion_per["PER_Base"].notna().sum()))
print("PER reproducido disponible:",
      int(comparacion_per["PER_Control"].notna().sum()))
print("Diferencia máxima original vs. reproducido (USD):",
      diferencia_control.max())

# Solo comparar valores disponibles en ambos escenarios
ambos = comparacion_per[
    ["PER_Control", "PER_Sin_Contradicciones"]
].notna().all(axis=1)

comparacion_per["Variacion_PER_pct"] = np.nan
comparacion_per.loc[ambos, "Variacion_PER_pct"] = (
    (
        comparacion_per.loc[ambos, "PER_Sin_Contradicciones"]
        / comparacion_per.loc[ambos, "PER_Control"]
    ) - 1
) * 100

print("\n=== IMPACTO DE EXCLUIR CONTRADICCIONES ===")
print("PER alternativo disponible:",
      int(comparacion_per["PER_Sin_Contradicciones"].notna().sum()))
print("Propiedades comparables en ambos escenarios:",
      int(ambos.sum()))
print("Propiedades con variación mayor a 0,01%:",
      int((comparacion_per["Variacion_PER_pct"].abs() > 0.01).sum()))
print("Variación absoluta mediana (%):",
      round(comparacion_per["Variacion_PER_pct"].abs().median(), 4))
print("Variación absoluta máxima (%):",
      round(comparacion_per["Variacion_PER_pct"].abs().max(), 4))


=== CONTROL DE REPRODUCIBILIDAD ===
PER original disponible: 428
PER reproducido disponible: 428
Diferencia máxima original vs. reproducido (USD): 1.1641532182693481e-10

=== IMPACTO DE EXCLUIR CONTRADICCIONES ===
PER alternativo disponible: 428
Propiedades comparables en ambos escenarios: 428
Propiedades con variación mayor a 0,01%: 35
Variación absoluta mediana (%): 0.0
Variación absoluta máxima (%): 1.4692


In [13]:

# 2.3 — Análisis de las propiedades cuyo PER cambió

# Umbral para distinguir cambios relevantes de diferencias numéricas
UMBRAL_CAMBIO = 0.01

afectadas = comparacion_per.loc[
    comparacion_per["Variacion_PER_pct"].abs() > UMBRAL_CAMBIO
].copy()

afectadas["Variacion_Absoluta_pct"] = (
    afectadas["Variacion_PER_pct"].abs()
)

afectadas["Diferencia_USD"] = (
    afectadas["PER_Sin_Contradicciones"]
    - afectadas["PER_Control"]
)

print("=== RESUMEN DE PROPIEDADES AFECTADAS ===")
print("Cantidad:", len(afectadas))
print("Porcentaje sobre las 428 evaluadas:",
      round(len(afectadas) / 428 * 100, 2), "%")

print("\nVariación porcentual:")
display(
    afectadas["Variacion_PER_pct"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95])
    .round(3)
)

print("\nVariación en dólares:")
display(
    afectadas["Diferencia_USD"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95])
    .round(2)
)

print("\n=== BARRIOS MÁS AFECTADOS ===")
display(
    afectadas.groupby("Barrio_Oficial").agg(
        Propiedades_afectadas=("ID", "count"),
        Variacion_media_abs_pct=("Variacion_Absoluta_pct", "mean"),
        Variacion_max_abs_pct=("Variacion_Absoluta_pct", "max")
    )
    .sort_values("Propiedades_afectadas", ascending=False)
    .round(3)
)

print("\n=== 10 MAYORES CAMBIOS DEL PER ===")
display(
    afectadas.sort_values(
        "Variacion_Absoluta_pct", ascending=False
    )[
        [
            "ID",
            "Barrio_Oficial",
            "Ambientes",
            "PER_Control",
            "PER_Sin_Contradicciones",
            "Diferencia_USD",
            "Variacion_PER_pct"
        ]
    ].head(10).round(2)
)


=== RESUMEN DE PROPIEDADES AFECTADAS ===
Cantidad: 35
Porcentaje sobre las 428 evaluadas: 8.18 %

Variación porcentual:


,Variacion_PER_pct
count,35.000
mean,0.745
std,0.576
min,-0.169
25%,0.272
50%,0.721
75%,1.101
95%,1.469
max,1.469



Variación en dólares:


,Diferencia_USD
count,35.00
mean,675.87
std,604.07
min,-298.95
25%,196.29
50%,649.79
75%,1232.67
95%,1483.93
max,1969.18



=== BARRIOS MÁS AFECTADOS ===


,Propiedades_afectadas,Variacion_media_abs_pct,Variacion_max_abs_pct
Barrio_Oficial,,,
Almagro,9,0.935,1.048
Caballito,5,0.808,1.155
San Nicolas,4,1.469,1.469
Colegiales,3,0.413,0.535
San Cristobal,3,0.012,0.012
Nuñez,3,0.169,0.169
Retiro,2,1.469,1.469
Flores,2,0.015,0.015
Boedo,1,0.867,0.867



=== 10 MAYORES CAMBIOS DEL PER ===


,ID,Barrio_Oficial,Ambientes,PER_Control,PER_Sin_Contradicciones,Diferencia_USD,Variacion_PER_pct
4838,612524,San Nicolas,2,83898.31,85130.97,1232.67,1.47
6231,594011,San Nicolas,2,83898.31,85130.97,1232.67,1.47
1794,644370,Monserrat,2,87254.24,88536.21,1281.97,1.47
5138,608953,Constitucion,2,91449.15,92792.76,1343.61,1.47
2946,633218,Retiro,2,97640.85,99075.42,1434.58,1.47
4640,615214,San Nicolas,2,134027.54,135996.73,1969.18,1.47
2144,640951,Retiro,2,83780.85,85011.79,1230.94,1.47
5735,601661,San Nicolas,2,100266.86,101740.02,1473.16,1.47
2841,634324,Caballito,1,65800.90,66560.66,759.76,1.15
4304,619055,Almagro,2,57606.33,58210.20,603.87,1.05


## 2.4. Impacto sobre la identificación de oportunidades

Se evalúa si las variaciones del PER producidas por la exclusión
de referencias contradictorias modifican la clasificación de
oportunidades bajo un umbral de descuento bruto del 25%.

Este análisis no reemplaza la evaluación de rentabilidad neta:
permite determinar si la selección de comparables altera la
decisión preliminar de identificar una propiedad como candidata.

In [15]:

# 2.4.1 — Validación del precio publicado

print("=== PRECIO PUBLICADO ===")

display(
    df.loc[mask_objetivo_per, "Precio_Valor"]
    .describe()
    .round(2)
)

print("\n=== MONEDAS DISPONIBLES ===")
print([
    c for c in df.columns
    if "moneda" in c.lower()
])

print("\n=== EJEMPLOS DE PROPIEDADES CON PER ===")
display(
    df.loc[
        mask_objetivo_per & df["PER_Base"].notna(),
        ["ID", "Barrio_Oficial", "Precio_Valor", "PER_Base"]
    ].head(10)
)


=== PRECIO PUBLICADO ===


,Precio_Valor
count,439.00
mean,135712.79
std,144188.16
min,23000.00
25%,60000.00
50%,89000.00
75%,148500.00
max,1150000.00



=== MONEDAS DISPONIBLES ===
['Moneda', 'Moneda_Expensas']

=== EJEMPLOS DE PROPIEDADES CON PER ===


,ID,Barrio_Oficial,Precio_Valor,PER_Base
18,659191,Palermo,63000.0,61298.437192
63,658750,Balvanera,79500.0,81436.538462
97,658536,Recoleta,72000.0,99990.344901
162,657958,Recoleta,95000.0,132777.197827
212,657683,San Telmo,55000.0,57500.000000
222,657559,San Nicolas,47000.0,63516.208663
257,657288,Villa Pueyrredon,69000.0,141575.534247
266,657180,Belgrano,385000.0,428710.262222
278,657035,Palermo,1150000.0,816717.888936
300,656763,Balvanera,47000.0,59358.618563


In [16]:

# 2.4.2 — Sensibilidad de oportunidades al PER alternativo

UMBRAL_DESCUENTO = 0.25

# Comprobar moneda en el universo evaluado
universo = mask_objetivo_per & df["PER_Base"].notna()

print("=== MONEDAS DEL UNIVERSO EVALUADO ===")
display(df.loc[universo, "Moneda"].value_counts(dropna=False))

monedas = (
    df.loc[universo, "Moneda"]
    .astype(str).str.upper().str.strip()
)

if not monedas.isin(["USD", "U$S", "US$"]).all():
    print("\nREVISAR: hay monedas distintas de USD.")
    print("No se calcula el descuento hasta verificar la moneda.")

else:
    # Incorporar precio publicado
    comparacion_per["Precio_Publicado_USD"] = df.loc[
        mask_objetivo_per, "Precio_Valor"
    ]

    # Solo comparar casos con PER en ambos escenarios
    validos = (
        comparacion_per["PER_Control"].gt(0)
        & comparacion_per["PER_Sin_Contradicciones"].gt(0)
        & comparacion_per["Precio_Publicado_USD"].gt(0)
    )

    comp = comparacion_per.loc[validos].copy()

    comp["Descuento_Original"] = (
        1 - comp["Precio_Publicado_USD"] / comp["PER_Control"]
    )

    comp["Descuento_Alternativo"] = (
        1 - comp["Precio_Publicado_USD"]
        / comp["PER_Sin_Contradicciones"]
    )

    comp["Candidata_Original"] = (
        comp["Descuento_Original"] >= UMBRAL_DESCUENTO
    )

    comp["Candidata_Alternativa"] = (
        comp["Descuento_Alternativo"] >= UMBRAL_DESCUENTO
    )

    comp["Cambio_Clasificacion"] = (
        comp["Candidata_Original"]
        != comp["Candidata_Alternativa"]
    )

    print("\n=== IMPACTO SOBRE OPORTUNIDADES ===")
    print("Propiedades evaluadas:", len(comp))
    print("Candidatas originales:",
          int(comp["Candidata_Original"].sum()))
    print("Candidatas alternativas:",
          int(comp["Candidata_Alternativa"].sum()))
    print("Propiedades que cambian de clasificación:",
          int(comp["Cambio_Clasificacion"].sum()))

    print("\n=== MATRIZ DE CAMBIOS ===")
    display(pd.crosstab(
        comp["Candidata_Original"],
        comp["Candidata_Alternativa"],
        rownames=["Original"],
        colnames=["Sin contradicciones"]
    ))

    print("\n=== PROPIEDADES RECLASIFICADAS ===")
    display(
        comp.loc[comp["Cambio_Clasificacion"], [
            "ID", "Barrio_Oficial", "Precio_Publicado_USD",
            "PER_Control", "PER_Sin_Contradicciones",
            "Descuento_Original", "Descuento_Alternativo"
        ]].round(4)
    )


=== MONEDAS DEL UNIVERSO EVALUADO ===


,count
Moneda,
USD,428



=== IMPACTO SOBRE OPORTUNIDADES ===
Propiedades evaluadas: 428
Candidatas originales: 162
Candidatas alternativas: 162
Propiedades que cambian de clasificación: 0

=== MATRIZ DE CAMBIOS ===


Sin contradicciones,False,True
Original,,
False,266,0
True,0,162



=== PROPIEDADES RECLASIFICADAS ===


,ID,Barrio_Oficial,Precio_Publicado_USD,PER_Control,PER_Sin_Contradicciones,Descuento_Original,Descuento_Alternativo



### 2.5. Conclusión: robustez del PER ante referencias contradictorias

Se identificaron 78 publicaciones con señales contradictorias sobre su estado, de las cuales 10 formaban parte del conjunto de 2.844 referencias potenciales del Precio Esperado Reciclado (PER).

Al excluir estas 10 referencias y recalcular el PER con la misma metodología jerárquica, la cobertura permaneció constante en 428 de las 439 propiedades que necesitan obra (97,49%).

De las 428 propiedades evaluadas, 35 (8,18%) presentaron variaciones superiores al 0,01% en su PER. La variación relativa máxima fue de 1,47%, mientras que la mediana de variación absoluta sobre el conjunto completo fue 0%.

Al aplicar un umbral preliminar de descuento bruto del 25%, se identificaron 162 candidatas tanto en el escenario original como en el alternativo, sin reclasificaciones.

**Conclusión:** el PER presenta estabilidad frente a la exclusión de las referencias contradictorias identificadas. Si bien algunas estimaciones individuales cambian, estos cambios no modifican la selección preliminar de oportunidades bajo el umbral evaluado.

Este resultado constituye evidencia de robustez ante una fuente específica de incertidumbre, pero no garantiza la precisión del PER frente a otras decisiones metodológicas ni implica rentabilidad neta de las operaciones de flipping.


## 3. Sensibilidad del PER a la inclusión de referencias con señales débiles de reciclado

Se evalúa cómo cambia el Precio Esperado Reciclado (PER) al incluir o excluir referencias que presentan exclusivamente señales textuales débiles de reciclado.

Para ello, se comparan dos escenarios:

- **Escenario restrictivo:** excluye las 12 referencias identificadas únicamente mediante señales débiles.
- **Escenario ampliado:** conserva esas 12 referencias dentro del conjunto original de comparables.

Ambos escenarios mantienen los restantes criterios de selección y utilizan la misma metodología de benchmark jerárquico.

El objetivo es evaluar el impacto sobre la cobertura del PER, las estimaciones obtenidas y la identificación preliminar de oportunidades de inversión.

Este análisis complementa la notebook 03 y permite evaluar la sensibilidad del benchmark ante una modificación puntual en la composición de sus referencias, sin reconstruir completamente el modelo de clasificación original.

In [19]:

# 3.3 — Identificación preliminar de señales de reciclado

import re
import unicodedata

def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caracter for caracter in texto
        if not unicodedata.combining(caracter)
    )
    return re.sub(r"\s+", " ", texto).strip()

texto_publicacion = (
    df["Titulo"].fillna("").astype(str)
    + " "
    + df["Descripcion"].fillna("").astype(str)
).map(normalizar_texto)

# Patrones existentes en src.patrones
patron_fuerte = PATRONES["reciclada"]
patron_debil = PATRONES["reciclada_debil"]

# Detección textual preliminar (sin aplicar todavía negaciones)
senal_fuerte = texto_publicacion.str.contains(
    patron_fuerte, regex=True, na=False
)

senal_debil = texto_publicacion.str.contains(
    patron_debil, regex=True, na=False
)

resumen_senales = pd.DataFrame({
    "Grupo": [
        "Solo señal fuerte",
        "Solo señal débil",
        "Ambas señales",
        "Ninguna señal"
    ],
    "Cantidad": [
        int((senal_fuerte & ~senal_debil).sum()),
        int((~senal_fuerte & senal_debil).sum()),
        int((senal_fuerte & senal_debil).sum()),
        int((~senal_fuerte & ~senal_debil).sum())
    ]
})

print("=== SEÑALES TEXTUALES PRELIMINARES ===")
display(resumen_senales)

print("\n=== COMPARACIÓN CON RECICLADO_TEXTO ===")
display(pd.crosstab(
    df["Reciclado_Texto"],
    senal_fuerte | senal_debil,
    rownames=["Reciclado_Texto original"],
    colnames=["Señal textual preliminar"],
    margins=True
))

print("\n=== SEÑALES ENTRE REFERENCIAS DEL PER ===")
display(pd.DataFrame({
    "Indicador": [
        "Referencias originales",
        "Con señal fuerte",
        "Con señal débil",
        "Con alguna señal"
    ],
    "Cantidad": [
        int(mask_ref_original.sum()),
        int((mask_ref_original & senal_fuerte).sum()),
        int((mask_ref_original & senal_debil).sum()),
        int((mask_ref_original & (senal_fuerte | senal_debil)).sum())
    ]
}))


=== SEÑALES TEXTUALES PRELIMINARES ===


,Grupo,Cantidad
0,Solo señal fuerte,1081
1,Solo señal débil,14
2,Ambas señales,6
3,Ninguna señal,8398



=== COMPARACIÓN CON RECICLADO_TEXTO ===


Señal textual preliminar,False,True,All
Reciclado_Texto original,,,
False,8398,14,8412
True,0,1087,1087
All,8398,1101,9499



=== SEÑALES ENTRE REFERENCIAS DEL PER ===


,Indicador,Cantidad
0,Referencias originales,2844
1,Con señal fuerte,804
2,Con señal débil,18
3,Con alguna señal,816


In [20]:

# 3.4 — Señales de reciclado por estado de propiedad

diagnostico = df.loc[mask_ref_original, [
    "Estado_Propiedad", "Estado_Agrupado",
    "Reciclado_Texto", "Reciclado_Parcial"
]].copy()

diagnostico["Senal_Fuerte"] = senal_fuerte.loc[mask_ref_original]
diagnostico["Senal_Debil"] = senal_debil.loc[mask_ref_original]

diagnostico["Tipo_Senal"] = np.select(
    [
        diagnostico["Senal_Fuerte"] & diagnostico["Senal_Debil"],
        diagnostico["Senal_Fuerte"],
        diagnostico["Senal_Debil"]
    ],
    [
        "Ambas",
        "Solo fuerte",
        "Solo debil"
    ],
    default="Ninguna"
)

print("=== ESTADOS DE LAS REFERENCIAS ===")
display(pd.crosstab(
    diagnostico["Estado_Propiedad"],
    diagnostico["Tipo_Senal"],
    margins=True
))

print("\n=== REFERENCIAS CON SOLO SEÑAL DÉBIL ===")
display(
    df.loc[
        mask_ref_original & senal_debil & ~senal_fuerte,
        ["ID", "Barrio_Oficial", "Estado_Propiedad",
         "Reciclado_Texto", "Titulo"]
    ]
)


=== ESTADOS DE LAS REFERENCIAS ===


Tipo_Senal,Ambas,Ninguna,Solo debil,Solo fuerte,All
Estado_Propiedad,,,,,
Buen_estado,0,1884,7,0,1891
Reciclada_refaccionada,6,144,5,798,953
All,6,2028,12,798,2844



=== REFERENCIAS CON SOLO SEÑAL DÉBIL ===


,ID,Barrio_Oficial,Estado_Propiedad,Reciclado_Texto,Titulo
28,659112,Almagro,Buen_estado,False,VENTA DEPARTAMENTO 2 AMBIENTES EN ALMAGRO
43,658956,Balvanera,Buen_estado,False,VENTA DEPTO 3 AMB P.BAJA C/PATIO APTO CRED
598,654482,Almagro,Buen_estado,False,VENTA MONOAMBIENTE CON BALCON VISTA ABIERTA
1690,645225,Recoleta,Buen_estado,False,VENTA DEPARTAMENTO 2 AMBIENTES BARRIO NORTE
3342,629386,Caballito,Reciclada_refaccionada,False,Dúplex en Caballito con balcón cerrado y cochera
3530,627664,Caballito,Buen_estado,False,CABALLITO NORTE DEPARTAMENTO VENTA 2 AMBIENTES
3543,627609,Caballito,Reciclada_refaccionada,False,Venta 4 ambientes con dependecia Caballito
3814,624686,Villa Urquiza,Buen_estado,False,VENTA DEPARTAMENTO 2 AMBIENTES VILLA URQUIZA
4948,611305,Parque Chacabuco,Reciclada_refaccionada,False,Departamento ideal para empezar a ser propiet...
5044,610234,Villa Devoto,Reciclada_refaccionada,False,DEPARTAMENTO PISO 4 AMB VENTA VILLA DEVOTO


In [21]:

# 3.5 — Escenarios de sensibilidad a señales débiles

# Base: referencias originales, sin señales contradictorias
mask_base = mask_ref_sin_contradicciones.copy()

# Referencias que presentan solo señal débil
mask_solo_debil = senal_debil & ~senal_fuerte

# Escenario restrictivo: excluir referencias con solo señal débil
mask_escenario_fuerte = mask_base & ~mask_solo_debil

# Escenario ampliado: conservar esas referencias
mask_escenario_ampliado = mask_base.copy()

print("=== TAMAÑO DE LOS ESCENARIOS ===")
print("Referencias base:", int(mask_base.sum()))
print("Escenario restrictivo:", int(mask_escenario_fuerte.sum()))
print("Escenario ampliado:", int(mask_escenario_ampliado.sum()))

print("\n=== REFERENCIAS EXCLUIDAS ===")
display(
    df.loc[
        mask_base & mask_solo_debil,
        ["ID", "Barrio_Oficial", "Estado_Propiedad",
         "Reciclado_Texto", "Titulo"]
    ]
)

print("\n=== CONTROL DE DIFERENCIAS ===")
print(
    "Referencias que cambian entre escenarios:",
    int((mask_escenario_fuerte != mask_escenario_ampliado).sum())
)


=== TAMAÑO DE LOS ESCENARIOS ===
Referencias base: 2834
Escenario restrictivo: 2822
Escenario ampliado: 2834

=== REFERENCIAS EXCLUIDAS ===


,ID,Barrio_Oficial,Estado_Propiedad,Reciclado_Texto,Titulo
28,659112,Almagro,Buen_estado,False,VENTA DEPARTAMENTO 2 AMBIENTES EN ALMAGRO
43,658956,Balvanera,Buen_estado,False,VENTA DEPTO 3 AMB P.BAJA C/PATIO APTO CRED
598,654482,Almagro,Buen_estado,False,VENTA MONOAMBIENTE CON BALCON VISTA ABIERTA
1690,645225,Recoleta,Buen_estado,False,VENTA DEPARTAMENTO 2 AMBIENTES BARRIO NORTE
3342,629386,Caballito,Reciclada_refaccionada,False,Dúplex en Caballito con balcón cerrado y cochera
3530,627664,Caballito,Buen_estado,False,CABALLITO NORTE DEPARTAMENTO VENTA 2 AMBIENTES
3543,627609,Caballito,Reciclada_refaccionada,False,Venta 4 ambientes con dependecia Caballito
3814,624686,Villa Urquiza,Buen_estado,False,VENTA DEPARTAMENTO 2 AMBIENTES VILLA URQUIZA
4948,611305,Parque Chacabuco,Reciclada_refaccionada,False,Departamento ideal para empezar a ser propiet...
5044,610234,Villa Devoto,Reciclada_refaccionada,False,DEPARTAMENTO PISO 4 AMB VENTA VILLA DEVOTO



=== CONTROL DE DIFERENCIAS ===
Referencias que cambian entre escenarios: 12


In [22]:

# 3.6 — Recalcular el PER bajo ambos escenarios

per_restrictivo = benchmark_jerarquico(
    df,
    valor="USD_m2_Homogeneizado",
    niveles=NIVELES_COMPARABLES,
    mask_referencia=mask_escenario_fuerte,
    mask_objetivo=mask_objetivo_per,
    min_n=MIN_COMPARABLES,
    leave_one_out=False
)

per_ampliado = benchmark_jerarquico(
    df,
    valor="USD_m2_Homogeneizado",
    niveles=NIVELES_COMPARABLES,
    mask_referencia=mask_escenario_ampliado,
    mask_objetivo=mask_objetivo_per,
    min_n=MIN_COMPARABLES,
    leave_one_out=False
)

# Construcción de la tabla comparativa
sensibilidad_senales = df.loc[
    mask_objetivo_per,
    ["ID", "Barrio_Oficial", "Precio_Valor",
     "Superficie_Homogeneizada_m2"]
].copy()

sensibilidad_senales["PER_Restrictivo"] = (
    per_restrictivo.loc[mask_objetivo_per, "Q50"]
    * sensibilidad_senales["Superficie_Homogeneizada_m2"]
)

sensibilidad_senales["PER_Ampliado"] = (
    per_ampliado.loc[mask_objetivo_per, "Q50"]
    * sensibilidad_senales["Superficie_Homogeneizada_m2"]
)

# Comparación solo entre propiedades con ambos PER disponibles
validos = (
    sensibilidad_senales["PER_Restrictivo"].gt(0)
    & sensibilidad_senales["PER_Ampliado"].gt(0)
)

sensibilidad_senales["Variacion_PER_pct"] = np.nan

sensibilidad_senales.loc[validos, "Variacion_PER_pct"] = (
    (
        sensibilidad_senales.loc[validos, "PER_Ampliado"]
        / sensibilidad_senales.loc[validos, "PER_Restrictivo"]
    ) - 1
) * 100

print("=== COBERTURA DEL PER ===")
print("Propiedades que necesitan obra:", int(mask_objetivo_per.sum()))
print("PER restrictivo disponible:",
      int(sensibilidad_senales["PER_Restrictivo"].notna().sum()))
print("PER ampliado disponible:",
      int(sensibilidad_senales["PER_Ampliado"].notna().sum()))
print("Propiedades comparables en ambos escenarios:",
      int(validos.sum()))

print("\n=== VARIACIÓN DEL PER (%) ===")
display(
    sensibilidad_senales.loc[validos, "Variacion_PER_pct"]
    .describe()
    .round(4)
    .to_frame("Variacion_PER_pct")
)

print("\n=== IMPACTO SOBRE LAS ESTIMACIONES ===")
variacion = sensibilidad_senales.loc[validos, "Variacion_PER_pct"].abs()

print("Sin cambios relevantes (<= 0,01%):",
      int((variacion <= 0.01).sum()))
print("Con cambios (> 0,01%):",
      int((variacion > 0.01).sum()))
print("Variación absoluta máxima (%):",
      round(variacion.max(), 4))
print("Variación absoluta mediana (%):",
      round(variacion.median(), 4))


=== COBERTURA DEL PER ===
Propiedades que necesitan obra: 439
PER restrictivo disponible: 428
PER ampliado disponible: 428
Propiedades comparables en ambos escenarios: 428

=== VARIACIÓN DEL PER (%) ===


,Variacion_PER_pct
count,428.0000
mean,-0.0429
std,0.5609
min,-2.8690
25%,0.0000
50%,0.0000
75%,0.0000
max,2.9083



=== IMPACTO SOBRE LAS ESTIMACIONES ===
Sin cambios relevantes (<= 0,01%): 383
Con cambios (> 0,01%): 45
Variación absoluta máxima (%): 2.9083
Variación absoluta mediana (%): 0.0


In [23]:

# 3.7 — Impacto de las señales débiles sobre las oportunidades

UMBRAL_DESCUENTO = 0.25

comparacion_oportunidades = sensibilidad_senales.loc[
    validos
].copy()

comparacion_oportunidades["Descuento_Restrictivo"] = (
    1
    - comparacion_oportunidades["Precio_Valor"]
    / comparacion_oportunidades["PER_Restrictivo"]
)

comparacion_oportunidades["Descuento_Ampliado"] = (
    1
    - comparacion_oportunidades["Precio_Valor"]
    / comparacion_oportunidades["PER_Ampliado"]
)

comparacion_oportunidades["Candidata_Restrictivo"] = (
    comparacion_oportunidades["Descuento_Restrictivo"]
    >= UMBRAL_DESCUENTO
)

comparacion_oportunidades["Candidata_Ampliado"] = (
    comparacion_oportunidades["Descuento_Ampliado"]
    >= UMBRAL_DESCUENTO
)

comparacion_oportunidades["Cambio_Clasificacion"] = (
    comparacion_oportunidades["Candidata_Restrictivo"]
    != comparacion_oportunidades["Candidata_Ampliado"]
)

print("=== IMPACTO SOBRE OPORTUNIDADES ===")
print(
    "Candidatas escenario restrictivo:",
    int(comparacion_oportunidades["Candidata_Restrictivo"].sum())
)
print(
    "Candidatas escenario ampliado:",
    int(comparacion_oportunidades["Candidata_Ampliado"].sum())
)
print(
    "Propiedades reclasificadas:",
    int(comparacion_oportunidades["Cambio_Clasificacion"].sum())
)

print("\n=== MATRIZ DE CLASIFICACIÓN ===")
display(pd.crosstab(
    comparacion_oportunidades["Candidata_Restrictivo"],
    comparacion_oportunidades["Candidata_Ampliado"],
    rownames=["Restrictivo"],
    colnames=["Ampliado"]
))

print("\n=== PROPIEDADES QUE CAMBIAN DE CLASIFICACIÓN ===")
display(
    comparacion_oportunidades.loc[
        comparacion_oportunidades["Cambio_Clasificacion"],
        [
            "ID",
            "Barrio_Oficial",
            "Precio_Valor",
            "PER_Restrictivo",
            "PER_Ampliado",
            "Descuento_Restrictivo",
            "Descuento_Ampliado",
            "Variacion_PER_pct"
        ]
    ].round(4)
)


=== IMPACTO SOBRE OPORTUNIDADES ===
Candidatas escenario restrictivo: 165
Candidatas escenario ampliado: 162
Propiedades reclasificadas: 3

=== MATRIZ DE CLASIFICACIÓN ===


Ampliado,False,True
Restrictivo,,
False,263,0
True,3,162



=== PROPIEDADES QUE CAMBIAN DE CLASIFICACIÓN ===


,ID,Barrio_Oficial,Precio_Valor,PER_Restrictivo,PER_Ampliado,Descuento_Restrictivo,Descuento_Ampliado,Variacion_PER_pct
4228,619835,Balvanera,84000.0,112041.7951,108827.2996,0.2503,0.2281,-2.8690
7520,567437,Balvanera,37500.0,50968.8029,49757.2334,0.2643,0.2463,-2.3771
7521,567435,Balvanera,36000.0,48787.8218,47628.0961,0.2621,0.2441,-2.3771



### 3.8. Conclusión: sensibilidad a señales débiles de reciclado

Se evaluó la sensibilidad del Precio Esperado Reciclado (PER) ante la inclusión de referencias que presentan exclusivamente señales textuales débiles de reciclado.

Se compararon dos escenarios:

- **Restrictivo:** 2.822 referencias, excluyendo las 12 publicaciones con señales exclusivamente débiles.
- **Ampliado:** 2.834 referencias, conservando esas publicaciones.

Ambos escenarios mantuvieron los restantes criterios de selección de comparables y utilizaron la misma metodología de benchmark jerárquico.

La cobertura del PER permaneció constante en 428 de las 439 propiedades clasificadas como `Necesita_obra` (97,49%).

De las 428 estimaciones comparables, 45 (10,51%) presentaron variaciones superiores al 0,01%. La variación absoluta máxima fue de 2,91%, mientras que la mediana fue del 0%.

Al aplicar un umbral preliminar de descuento bruto del 25%, el escenario restrictivo identificó 165 candidatas y el ampliado identificó 162.

**Tres propiedades cambiaron de clasificación**, todas ubicadas en Balvanera. En los tres casos, la inclusión de señales débiles redujo el PER estimado y el descuento bruto quedó por debajo del umbral establecido.

**Conclusión:** el benchmark presenta estabilidad general frente a esta modificación, pero algunas decisiones de selección de oportunidades son sensibles a la composición de las referencias, especialmente cuando el descuento estimado se encuentra cerca del umbral del 25%.

Por lo tanto, las propiedades próximas al umbral deben interpretarse con mayor cautela y revisarse antes de incorporarlas a un ranking definitivo de inversión.

**Alcance:** esta prueba analiza la inclusión o exclusión de 12 referencias con señales exclusivamente débiles, manteniendo el resto de la clasificación original. No representa una comparación integral entre dos modelos alternativos de clasificación de propiedades.



## 4. Incertidumbre asociada a propiedades sin clasificar

Una limitación del análisis es la proporción de publicaciones cuyo estado de conservación no pudo determinarse mediante las reglas de clasificación textual.

De las 9.499 propiedades analizadas, 6.034 (63,52%) fueron clasificadas como `Sin_clasificar`.

Esta sección evalúa si las propiedades sin clasificar presentan diferencias en su distribución geográfica y características respecto de las propiedades clasificadas.

El objetivo es identificar posibles sesgos de selección que puedan afectar la representatividad de los comparables utilizados para estimar el Precio Esperado Reciclado (PER).

La falta de clasificación no implica necesariamente que las propiedades estén en mal estado: representa incertidumbre sobre su condición.


In [24]:

# 4.1 — Diagnóstico de propiedades sin clasificar

mask_sin_clasificar = df["Estado_Propiedad"].eq("Sin_clasificar")

print("=== DISTRIBUCIÓN DE ESTADOS ===")

resumen_estados = (
    df["Estado_Propiedad"]
    .value_counts(dropna=False)
    .rename_axis("Estado")
    .reset_index(name="Cantidad")
)

resumen_estados["Porcentaje"] = (
    resumen_estados["Cantidad"] / len(df) * 100
).round(2)

display(resumen_estados)

print("\n=== PROPIEDADES SIN CLASIFICAR POR BARRIO ===")

resumen_barrios = (
    df.groupby("Barrio_Oficial", dropna=False)
    .agg(
        Total=("ID", "size"),
        Sin_clasificar=(
            "Estado_Propiedad",
            lambda x: x.eq("Sin_clasificar").sum()
        )
    )
    .reset_index()
)

resumen_barrios["Pct_Sin_Clasificar"] = (
    resumen_barrios["Sin_clasificar"]
    / resumen_barrios["Total"] * 100
).round(2)

# Evitamos interpretar porcentajes de barrios con muy pocos avisos
barrios_relevantes = (
    resumen_barrios[resumen_barrios["Total"] >= 50]
    .sort_values("Pct_Sin_Clasificar", ascending=False)
)

print("Barrios con al menos 50 publicaciones:")
print("Cantidad:", len(barrios_relevantes))

print("\n=== 10 BARRIOS CON MAYOR % SIN CLASIFICAR ===")
display(barrios_relevantes.head(10))

print("\n=== 10 BARRIOS CON MENOR % SIN CLASIFICAR ===")
display(barrios_relevantes.tail(10).sort_values("Pct_Sin_Clasificar"))


=== DISTRIBUCIÓN DE ESTADOS ===


,Estado,Cantidad,Porcentaje
0,Sin_clasificar,6034,63.52
1,Buen_estado,2049,21.57
2,Reciclada_refaccionada,977,10.29
3,Necesita_obra,439,4.62



=== PROPIEDADES SIN CLASIFICAR POR BARRIO ===
Barrios con al menos 50 publicaciones:
Cantidad: 39

=== 10 BARRIOS CON MAYOR % SIN CLASIFICAR ===


,Barrio_Oficial,Total,Sin_clasificar,Pct_Sin_Clasificar
41,Villa Ortuzar,106,89,83.96
40,Villa Luro,74,60,81.08
23,Parque Chas,51,40,78.43
29,Saavedra,152,115,75.66
47,Villa Urquiza,340,253,74.41
37,Villa Devoto,185,134,72.43
17,Monte Castro,107,77,71.96
8,Coghlan,64,46,71.88
19,Nuñez,265,189,71.32
36,Villa Del Parque,197,140,71.07



=== 10 BARRIOS CON MENOR % SIN CLASIFICAR ===


,Barrio_Oficial,Total,Sin_clasificar,Pct_Sin_Clasificar
3,Barracas,107,54,50.47
28,Retiro,215,109,50.70
39,Villa Lugano,101,54,53.47
27,Recoleta,654,356,54.43
12,Floresta,100,55,55.00
10,Constitucion,189,104,55.03
31,San Nicolas,286,162,56.64
15,Mataderos,65,37,56.92
2,Balvanera,612,359,58.66
33,Velez Sarsfield,60,36,60.00


In [25]:

# 4.2 — Comparación entre propiedades clasificadas y sin clasificar

variables_analisis = [
    "Superficie_Homogeneizada_m2",
    "Precio_Valor",
    "USD_m2_Homogeneizado"
]

df_comparacion = df.copy()

df_comparacion["Grupo_Clasificacion"] = np.where(
    df_comparacion["Estado_Propiedad"].eq("Sin_clasificar"),
    "Sin clasificar",
    "Clasificadas"
)

# Comparamos precios solo en publicaciones expresadas en USD
df_comparacion = df_comparacion.loc[
    df_comparacion["Moneda"].eq("USD")
].copy()

print("=== TAMAÑO DE LOS GRUPOS (USD) ===")
display(
    df_comparacion["Grupo_Clasificacion"]
    .value_counts()
    .to_frame("Cantidad")
)

print("\n=== MEDIANAS POR GRUPO ===")
medianas = (
    df_comparacion
    .groupby("Grupo_Clasificacion")[variables_analisis]
    .median()
    .round(2)
)
display(medianas)

print("\n=== DIFERENCIA RELATIVA DE MEDIANAS (%) ===")
diferencia_pct = (
    medianas.loc["Sin clasificar"]
    / medianas.loc["Clasificadas"] - 1
) * 100

display(
    diferencia_pct.round(2).to_frame("Diferencia_pct")
)

print("\n=== DISTRIBUCIÓN DEL PRECIO POR M² ===")
display(
    df_comparacion.groupby("Grupo_Clasificacion")[
        "USD_m2_Homogeneizado"
    ].describe(percentiles=[0.25, 0.5, 0.75]).round(2)
)


=== TAMAÑO DE LOS GRUPOS (USD) ===


,Cantidad
Grupo_Clasificacion,
Sin clasificar,6034
Clasificadas,3465



=== MEDIANAS POR GRUPO ===


,Superficie_Homogeneizada_m2,Precio_Valor,USD_m2_Homogeneizado
Grupo_Clasificacion,,,
Clasificadas,58.00,115000.0,2044.67
Sin clasificar,56.22,125000.0,2264.81



=== DIFERENCIA RELATIVA DE MEDIANAS (%) ===


,Diferencia_pct
Superficie_Homogeneizada_m2,-3.07
Precio_Valor,8.70
USD_m2_Homogeneizado,10.77



=== DISTRIBUCIÓN DEL PRECIO POR M² ===


,count,mean,std,min,25%,50%,75%,max
Grupo_Clasificacion,,,,,,,,
Clasificadas,3465.0,2115.90,692.77,297.44,1641.51,2044.67,2504.67,7271.82
Sin clasificar,6034.0,2417.58,961.55,385.60,1784.96,2264.81,2841.96,16755.32


In [26]:

# 4.3 — Diferencias de precio por m² controlando por barrio

comparacion_barrio = (
    df_comparacion
    .groupby(["Barrio_Oficial", "Grupo_Clasificacion"])
    .agg(
        N=("ID", "size"),
        Mediana_USD_m2=("USD_m2_Homogeneizado", "median")
    )
    .reset_index()
)

tabla_barrio = comparacion_barrio.pivot(
    index="Barrio_Oficial",
    columns="Grupo_Clasificacion",
    values=["N", "Mediana_USD_m2"]
)

tabla_barrio.columns = [
    f"{variable}_{grupo}"
    for variable, grupo in tabla_barrio.columns
]

tabla_barrio = tabla_barrio.reset_index()

# Exigimos al menos 20 propiedades en cada grupo
barrios_comparables = tabla_barrio.loc[
    (tabla_barrio["N_Clasificadas"] >= 20)
    & (tabla_barrio["N_Sin clasificar"] >= 20)
].copy()

barrios_comparables["Diferencia_pct"] = (
    barrios_comparables["Mediana_USD_m2_Sin clasificar"]
    / barrios_comparables["Mediana_USD_m2_Clasificadas"]
    - 1
) * 100

print("=== RESUMEN POR BARRIO ===")
print("Barrios comparables:", len(barrios_comparables))
print(
    "Barrios donde las sin clasificar son más caras por m²:",
    int((barrios_comparables["Diferencia_pct"] > 0).sum())
)
print(
    "Barrios donde las sin clasificar son más baratas por m²:",
    int((barrios_comparables["Diferencia_pct"] < 0).sum())
)

print("\n=== DISTRIBUCIÓN DE DIFERENCIAS (%) ===")
display(
    barrios_comparables["Diferencia_pct"]
    .describe()
    .round(2)
    .to_frame()
)

print("\n=== 10 BARRIOS CON MAYOR DIFERENCIA POSITIVA ===")
display(
    barrios_comparables.sort_values(
        "Diferencia_pct", ascending=False
    ).head(10).round(2)
)

print("\n=== 10 BARRIOS CON MAYOR DIFERENCIA NEGATIVA ===")
display(
    barrios_comparables.sort_values(
        "Diferencia_pct"
    ).head(10).round(2)
)


=== RESUMEN POR BARRIO ===
Barrios comparables: 35
Barrios donde las sin clasificar son más caras por m²: 33
Barrios donde las sin clasificar son más baratas por m²: 2

=== DISTRIBUCIÓN DE DIFERENCIAS (%) ===


,Diferencia_pct
count,35.00
mean,8.56
std,5.83
min,-2.48
25%,4.57
50%,8.18
75%,12.50
max,20.84



=== 10 BARRIOS CON MAYOR DIFERENCIA POSITIVA ===


,Barrio_Oficial,N_Clasificadas,N_Sin clasificar,Mediana_USD_m2_Clasificadas,Mediana_USD_m2_Sin clasificar,Diferencia_pct
32,San Telmo,40.0,76.0,1670.05,2018.01,20.84
22,Parque Chacabuco,56.0,88.0,1810.39,2185.73,20.73
19,Nuñez,76.0,189.0,2563.52,3023.47,17.94
7,Chacarita,38.0,79.0,2353.34,2764.54,17.47
30,San Cristobal,70.0,116.0,1585.03,1825.41,15.17
37,Villa Devoto,51.0,134.0,2160.00,2463.56,14.05
20,Palermo,389.0,738.0,2638.83,2988.47,13.25
42,Villa Pueyrredon,23.0,37.0,2002.50,2260.46,12.88
14,Liniers,26.0,58.0,1499.38,1689.76,12.70
28,Retiro,106.0,109.0,2110.87,2370.69,12.31



=== 10 BARRIOS CON MAYOR DIFERENCIA NEGATIVA ===


,Barrio_Oficial,N_Clasificadas,N_Sin clasificar,Mediana_USD_m2_Clasificadas,Mediana_USD_m2_Sin clasificar,Diferencia_pct
45,Villa Santa Rita,29.0,58.0,2070.18,2018.90,-2.48
27,Recoleta,298.0,356.0,2481.02,2457.17,-0.96
2,Balvanera,253.0,359.0,1571.43,1575.76,0.28
33,Velez Sarsfield,24.0,36.0,1604.28,1628.22,1.49
31,San Nicolas,124.0,162.0,1627.20,1654.61,1.68
35,Villa Crespo,110.0,239.0,2102.93,2164.18,2.91
11,Flores,168.0,265.0,1730.68,1786.89,3.25
39,Villa Lugano,47.0,54.0,886.70,917.34,3.46
15,Mataderos,28.0,37.0,1751.49,1825.70,4.24
4,Belgrano,201.0,306.0,2658.01,2788.59,4.91



### 4.4. Conclusión: incertidumbre por propiedades sin clasificar

De las 9.499 propiedades analizadas, 6.034 (63,52%) no pudieron clasificarse según su estado de conservación mediante las reglas textuales utilizadas.

Se identificaron diferencias relevantes en la proporción de propiedades sin clasificar entre barrios. Entre los barrios con al menos 50 publicaciones, el porcentaje varió desde 50,47% en Barracas hasta 83,96% en Villa Ortúzar.

También se observaron diferencias en las características de ambos grupos:

- La superficie homogeneizada mediana de las propiedades sin clasificar fue 3,07% menor.
- El precio publicado mediano fue 8,70% mayor.
- El precio homogeneizado mediano por m² fue 10,77% mayor.

Para evaluar si esta última diferencia podía explicarse exclusivamente por la composición geográfica, se realizó una comparación dentro de cada barrio, considerando únicamente aquellos con al menos 20 propiedades clasificadas y 20 sin clasificar.

En 33 de los 35 barrios evaluados (94,3%), las propiedades sin clasificar presentaron un precio mediano por m² superior al de las clasificadas. La mediana de las diferencias porcentuales entre barrios fue de +8,18%.

**Conclusión:** la falta de clasificación no se distribuye uniformemente entre barrios y está asociada con diferencias observables en los precios de publicación. Esto constituye una posible fuente de sesgo de selección para el conjunto de propiedades utilizadas como referencias del PER.

Sin embargo, estas diferencias no permiten cuantificar el sesgo del PER ni determinar su dirección, ya que no se conoce el estado real de las propiedades sin clasificar y pueden existir diferencias adicionales en superficie, ambientes, antigüedad y otras características.

Por lo tanto, las estimaciones del PER deben interpretarse como valores condicionados a las referencias disponibles y a las reglas de clasificación utilizadas, no como una representación exhaustiva del mercado.

Como mejora futura, se recomienda validar manualmente una muestra de propiedades sin clasificar y evaluar métodos adicionales de clasificación que permitan reducir la incertidumbre.



## 5. Síntesis de robustez y limitaciones

Los análisis realizados permitieron evaluar la estabilidad del Precio Esperado Reciclado (PER) frente a cambios en la selección de referencias y examinar las limitaciones de la clasificación textual de propiedades.

### 5.1. Resumen de resultados

| Evaluación | Resultado principal | Interpretación |
|---|---|---|
| Exclusión de referencias contradictorias | 10 referencias excluidas; 35 PER modificados; 0 reclasificaciones | Estabilidad frente a contradicciones identificadas |
| Sensibilidad a señales débiles | 12 referencias adicionales; 45 PER modificados; 3 reclasificaciones | Sensibilidad localizada, especialmente cerca del umbral |
| Cobertura del PER | 428 de 439 propiedades que necesitan obra (97,49%) | Cobertura estable en ambos análisis |
| Propiedades sin clasificar | 6.034 de 9.499 (63,52%) | Limitación importante de la clasificación textual |
| Diferencias de precios por barrio | 33 de 35 barrios con mayor precio mediano por m² entre las sin clasificar | Posible sesgo de selección, todavía no cuantificado |

### 5.2. Conclusión general

El PER mostró estabilidad general frente a las modificaciones evaluadas en el conjunto de referencias. En ambos análisis de sensibilidad se mantuvo la cobertura de 428 propiedades que necesitan obra.

Sin embargo, la inclusión de referencias con señales exclusivamente débiles modificó la clasificación preliminar de tres oportunidades ubicadas en Balvanera, lo que evidencia que las decisiones cercanas al umbral de descuento bruto del 25% requieren mayor cautela.

La principal limitación identificada es la elevada proporción de propiedades sin clasificar. Este grupo presenta diferencias sistemáticas de precios respecto de las propiedades clasificadas, incluso dentro de la mayoría de los barrios analizados.

En consecuencia, los resultados deben interpretarse como estimaciones condicionadas a la información disponible, a las reglas de clasificación y a la composición del conjunto de comparables.

### 5.3. Implicancias para la decisión de inversión

Se recomienda:

1. Mantener identificadas las oportunidades cercanas al umbral del 25% como casos sensibles a cambios metodológicos.
2. Complementar el descuento bruto con costos de refacción, gastos de transacción y rentabilidad neta antes de recomendar inversiones.
3. Validar manualmente una muestra de publicaciones para evaluar la calidad de las etiquetas de estado.
4. Documentar la incertidumbre asociada a propiedades sin clasificar y evitar interpretar la muestra etiquetada como representativa de todo el mercado.

**Conclusión final:** el PER constituye una herramienta de estimación y priorización preliminar de oportunidades, pero no reemplaza la validación del estado real del inmueble ni el análisis económico completo de una operación de flipping.
